# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [ ]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models. 

Instantiate a text generation pipeline using the tokenizer and model.

In [9]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [6]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Terve, miten menee?”\n”Hyvää kuuluu, kiitos kysymästä.”\n”Kuulin, että sinä olet ollut jonkin aikaa ilman äitiä.”'}]


Just print the text

In [7]:
print(output[0]['generated_text'])

Terve, miten menee?”
”Hyvää kuuluu, kiitos kysymästä.”
”Kuulin, että sinä olet ollut jonkin aikaa ilman äitiä.”


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [8]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Terve, miten menee? kysyy Sari kun istumme kahvipöydässä. Hän katselee ja koskettelua ihmetellen katsoo vuoroin miestä (sisukasta Sariksien vahvaa edustajaa jonka


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [13]:

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device)

def task_1():
    def zero_shot():
        prompt= """Do the following texts express a positive or negative sentiment?
                Text: I enjoyed hanging out with my friends.
                Answer:"""
        output = pipe(prompt, max_new_tokens=5)
        print(output[0]['generated_text'])

    def one_shot():
        prompt= """Do the following texts express a positive or negative sentiment?
                Text: I hated doing my homework.
                Answer: negative

                Text: I enjoyed hanging out with my friends.
                Answer:"""
        output = pipe(prompt, max_new_tokens=5)
        print(output[0]['generated_text'])

    def two_shot():
        prompt= """Do the following texts express a positive or negative sentiment?
                Text: I hated doing my homework.
                Answer: negative

                Text: The party was really fun.
                Answer: positive

                Text: I enjoyed hanging out with my friends.
                Answer:"""
        output = pipe(prompt, max_new_tokens=5)
        print(output[0]['generated_text'])

    zero_shot()
    one_shot()
    two_shot()

def task_2(): 
    def zero_shot(): 
        prompt= """List the person names occurring in the following texts.
                Text: John met Sarah at the university.
                Names:"""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text']) 
 
    def one_shot(): 
        prompt= """List the person names occurring in the following texts.
                Text: Alice talked to Robert yesterday.
                Names: Alice, Robert
 
                Text: John met Sarah at the university.
                Names:"""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text']) 
 
    def two_shot(): 
        prompt= """List the person names occurring in the following texts.
                Text: Alice talked to Robert yesterday.
                Names: Alice, Robert
 
                Text: James visited Emily last week.
                Names: James, Emily
 
                Text: John met Sarah at the university.
                Names:"""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text'])

    zero_shot()
    one_shot()
    two_shot()

def task_3(): 
    def zero_shot(): 
        prompt= """This is a first grade math exam.
                23 + 14 ="""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text']) 
 
    def one_shot(): 
        prompt= """This is a first grade math exam.
                11 + 22 = 33
 
                23 + 14 ="""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text']) 
 
    def two_shot(): 
        prompt= """This is a first grade math exam.
                11 + 22 = 33
 
                35 + 14 = 49
 
                23 + 14 ="""
        output = pipe(prompt, max_new_tokens=5) 
        print(output[0]['generated_text'])

    zero_shot()
    one_shot()
    two_shot()

task_1()
task_2()
task_3()

Loading weights: 100%|██████████| 436/436 [00:14<00:00, 29.11it/s] 
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gener

Do the following texts express a positive or negative sentiment?
                Text: I enjoyed hanging out with my friends.
                Answer: I enjoyed hanging out with


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Do the following texts express a positive or negative sentiment?
                Text: I hated doing my homework.
                Answer: negative

                Text: I enjoyed hanging out with my friends.
                Answer: negative

   


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

List the person names occurring in the following texts.
                Text: Alice talked to Robert yesterday.
                Names: Alice, Robert

                Text: James visited Emily last week.
                Names: James, Emily

                Text: John met Sarah at the university.
                Names: John, Sarah

 


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This is a first grade math exam.
                23 + 14 = 31    


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This is a first grade math exam.
                11 + 22 = 33

                23 + 14 = 28
   
This is a first grade math exam.
                11 + 22 = 33

                35 + 14 = 49

                23 + 14 = 23

   


**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**